In [1]:
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

projectPath = '/content/drive/MyDrive/TinyTransformer'
if not os.path.exists(projectPath):
  os.mkdir(projectPath)
%cd "$projectPath"

try:
  git_token = userdata.get('GITHUB_TOKEN')
except:
  print("Không lấy được TOKEN.")

!git config --global user.email "hhungln28@gmail.com"
!git config --global user.name "Hoang Hung"
!git branch -M main

username = "HoangHungLN"
repository = "TinyTransformer"
remote_url = f"https://{git_token}@github.com/{username}/{repository}.git"
!git remote set-url origin {remote_url}

Mounted at /content/drive
/content/drive/MyDrive/TinyTransformer


In [2]:
import torch
import torchvision
import torchvision.transforms as transform
from torch.utils.data import DataLoader

import sys
sys.path.append('/content/drive/MyDrive/TinyTransformer/src')
from config import TinyConfig

cfg = TinyConfig()
BATCH_SIZE = 32

# Preprocessing
transform = transform.Compose([
  transform.Resize((cfg.img_size, cfg.img_size)),
  transform.ToTensor(),
  transform.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Download dataset CIFAR-10
trainset = torchvision.datasets.CIFAR10(root = './data', train = True, download = True, transform = transform)
testset = torchvision.datasets.CIFAR10(root = './data', train = False, download = True, transform = transform)

dataloader = DataLoader(trainset, batch_size = BATCH_SIZE, shuffle = True)
testloader = DataLoader(testset, batch_size = BATCH_SIZE, shuffle = False)

In [3]:
import torch.nn as nn
from embeddings import PatchEmbed
from core import TinyTransformer

class TinyModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.embed = PatchEmbed(cfg)
    self.core = TinyTransformer(cfg)

  def forward(self, x):
    x = self.embed(x)
    x = self.core(x)
    return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TinyModel(cfg).to(device)

In [ ]:
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import os

checkpoint_path = 'weights/tiny_transformer_hw_aware_v3.pth'

if os.path.exists(checkpoint_path):
    print(f"Đang load model từ: {checkpoint_path}")
    # Load weights vào model hiện tại
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print("Load thành công! Bắt đầu Hardware-Aware Training.")
else:
    print(f"Không tìm thấy file {checkpoint_path}. Sẽ train lại từ đầu!")

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Dùng Scheduler để tự động giảm LR đi một nửa sau mỗi 5 Epochs
# Giúp model đi nhanh lúc đầu và len lỏi cẩn thận ở những epoch cuối
#scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

EXTRA_EPOCHS = 20

print(f"Fine-tuning với Base-2 Softmax trong {EXTRA_EPOCHS} epochs...")

for epoch in range(EXTRA_EPOCHS):
    running_loss = 0.0
    model.train()

    for i, data in enumerate(dataloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 200 == 199:
            print(f'[HW-Aware Epoch {epoch + 1}, Batch {i + 1}] Lỗi trung bình: {running_loss / 200:.4f}')
            running_loss = 0.0

    # Cập nhật Learning Rate cho Epoch tiếp theo
    # current_lr = optimizer.param_groups[0]['lr']
    # print(f"Hết Epoch {epoch + 1} | Learning Rate hiện tại: {current_lr:.6f}")
    # scheduler.step()

print('Đã Training xong phiên bản Phần cứng (Hardware-Aware)!')

# Lưu trọng số vào file mới để chuẩn bị cho Quantization
new_save_path = 'weights/tiny_transformer_hw_aware_v4.pth'
torch.save(model.state_dict(), new_save_path)
print(f"Đã lưu model tương thích phần cứng vào: {new_save_path}")

In [ ]:
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

all_preds = []
all_labels = []

model.eval()
print("Đang tổng hợp kết quả kiểm thử...")

with torch.no_grad():
    for data in testloader:
        images, labels = data
        images = images.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        # Chuyển từ Tensor (GPU) về List (CPU) để thư viện sklearn hiểu được
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print("\n--- BÁO CÁO CHI TIẾT (CLASSIFICATION REPORT) ---")
print(classification_report(all_labels, all_preds, target_names=classes))

print("\n--- MA TRẬN NHẦM LẪN (CONFUSION MATRIX) ---")
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.xlabel('Máy dự đoán (Predicted)')
plt.ylabel('Thực tế (Actual)')
plt.title('Confusion Matrix')
plt.show()

In [8]:
import torch
import numpy as np
import os
import math

out_dir = 'hex_weights_fixedpoint'
os.makedirs(out_dir, exist_ok=True)

checkpoint_path = 'weights/tiny_transformer_hw_aware_v3.pth'
state_dict = torch.load(checkpoint_path, map_location='cpu')

def float_to_fixed8_hex(val, fractional_bits):
  fixed_val = np.round(val * (2 ** fractional_bits))
  fixed_val = np.clip(fixed_val, -128, 127).astype(np.int8)
  # np.clip(): Nếu số lớn hơn 127 -> 127, bé hơn -128 -> -128, nằm trong đoạn -> giữ nguyên
  # astype(np.int8): Chuyển số về nhị phân int 8 bit có dấu
  hex_str = format(int(fixed_val) & 0xFF, '02X')
  return hex_str

print("Bắt đầu Lượng tử hóa Fixed-Point (Q-Format)...")

q_formats = {}
for layer_name, tensor in state_dict.items():
  if not tensor.is_floating_point():
    continue

  weight_np = tensor.numpy()
  max_abs_val = np.max(np.abs(weight_np))
  if max_abs_val == 0:
    fractional_bits = 7
  else:
    optimal_q = math.floor(math.log2(127.0 / max_abs_val))
    fractional_bits = max(0, min(7, optimal_q))

  q_formats[layer_name] = fractional_bits

  flat_weights = weight_np.flatten()
  safe_name = layer_name.replace('.', '_')
  file_path = os.path.join(out_dir, f"{safe_name}.hex")

  with open(file_path, 'w') as f:
    for w in flat_weights:
      f.write(f"{float_to_fixed8_hex(w, fractional_bits)}\n")

  print(f"Xuất: {safe_name}.hex | Định dạng: Q{7-fractional_bits}.{fractional_bits} (Fractional bits: {fractional_bits})")

print(f"\nHoàn tất! Đã lưu trong '{out_dir}/'")


Bắt đầu Lượng tử hóa Fixed-Point (Q-Format)...
Xuất: embed_proj_weight.hex | Định dạng: Q0.7 (Fractional bits: 7)
Xuất: embed_proj_bias.hex | Định dạng: Q0.7 (Fractional bits: 7)
Xuất: core_pos_embed.hex | Định dạng: Q1.6 (Fractional bits: 6)
Xuất: core_encoders_0_norm1_weight.hex | Định dạng: Q0.7 (Fractional bits: 7)
Xuất: core_encoders_0_norm1_bias.hex | Định dạng: Q0.7 (Fractional bits: 7)
Xuất: core_encoders_0_attn_qkv_weight.hex | Định dạng: Q0.7 (Fractional bits: 7)
Xuất: core_encoders_0_attn_proj_weight.hex | Định dạng: Q0.7 (Fractional bits: 7)
Xuất: core_encoders_0_attn_proj_bias.hex | Định dạng: Q0.7 (Fractional bits: 7)
Xuất: core_encoders_0_norm2_weight.hex | Định dạng: Q1.6 (Fractional bits: 6)
Xuất: core_encoders_0_norm2_bias.hex | Định dạng: Q0.7 (Fractional bits: 7)
Xuất: core_encoders_0_mlp_fc1_weight.hex | Định dạng: Q0.7 (Fractional bits: 7)
Xuất: core_encoders_0_mlp_fc1_bias.hex | Định dạng: Q0.7 (Fractional bits: 7)
Xuất: core_encoders_0_mlp_fc2_weight.hex | Định 

In [12]:
!git branch base-2Softmax
!git checkout base-2Softmax

M	src/TinyTransformer.ipynb
M	src/layers.py
Switched to branch 'base-2Softmax'


In [ ]:
!git add .
!git commit -m "implement base - Softmax and Quantization"
